# Step 6 — the SAM 2 temporal probe: is the sub-second gold instability label or scene?

**August plan, week 1.** Spec: A.4 in `local/tasks/roadmap-percepcion-rl.md`; design in
[[covt-reduced-sam-route]]. Not a rung — the ladder is closed. Trains nothing.

The counting gold moves **±0.86 between annotated frames less than a second apart** (1,946
pairs; the count changes in 56.6% of them). [[synthetic-counting-reconciled]] records that as an
**upper bound that cannot separate a real scene change from an annotation error**, and calls the
separation unavailable without clinical adjudication. At ≤1 s a SAM 2 track is the same physical
instance *by construction*, so:

> **track holds steady + gold jumps ≥3 = the gold is wrong.**

## Pre-registration — declared BEFORE the run

| result | reading | verdict |
|---|---|---|
| persistence J ≈ persistence S (`abs diff < EPS_PERSIST`) | the scene did **not** change where the gold jumped | 🟢 **LABEL** — annotation noise |
| persistence J **lower** than S | the scene really did change | 🔴 **SCENE** — the label is exonerated |
| persistence low in **both** | SAM 2 does not track this footage | ⚫ **NO VERDICT** |

**Arm J** = `abs(delta gold) >= JUMP`. **Arm S** = `gold_b == gold_a`, down-sampled to J's size
and **matched on the gap distribution** — gap is what drives track survival, so an unmatched
control would measure the sampling instead of the scene.

## The two blocking controls

🔴 **C1 — Rodrigo's negative control, restated.** His amendment: *"if SAM emits 20 masks where the
gold says 1, it does not discriminate and the whole reading dies."* The failure he names is real;
his literal threshold cannot be used, because **SAM 2 is class-agnostic** — one centre point on a
`heico` frame returns **90% of the image** (measured before this was written). `K >> 1` on a
`gold == 1` frame is a property of the tool. What must hold is **separation**: `gold >= 5` frames
must seed measurably more instances than `gold == 1` frames.
⚠️ **A C1 failure kills the count reading and this notebook reports NO VERDICT** — the
persistence-only fallback is deliberately *not* pre-registered as a rescue. Reading it after C1
fires would be moving the goalposts. **Rodrigo owns the amendment; the call is his.**

🔴 **C2 — tracker floor.** Persistence on **adjacent** frames (~40 ms) must be ≥ `C2_FLOOR`. If
SAM 2 cannot hold a track across 40 ms of our footage, neither arm means anything.

In [ ]:
# --- parameters (papermill) -----------------------------------------------------
# 🔴 Comments go ABOVE the assignment, never on the same line: papermill's parameter
# parser silently skips a line it cannot parse, `-p` is then ignored, and the notebook
# runs on its default. Measured 2026-08-05 on step 5.

SAM2 = "/workspace/models/sam2/sam2.1-hiera-large"
DATA_ROOT = "/workspace/orena-data"
# the by-construction regime: 31.1% of consecutive pairs sit inside 1 s
GAP_MAX = 1.0
# where our counting failure lives
MIN_GOLD = 5
# what counts as a jump, per ERROR_ANATOMY's own table
JUMP = 3
IOU_SURVIVE = 0.5
# the band within which the two arms are called equal. PRE-DECLARED.
EPS_PERSIST = 0.10
# below this in BOTH arms there is no verdict, only a statement about the tracker
LOW_PERSIST = 0.50
C2_FLOOR = 0.90
C1_MIN_SEPARATION = 0.5
GRID = 16
MIN_AREA_FRAC = 0.0005
MAX_AREA_FRAC = 0.25
NMS_IOU = 0.7
SEED = 42
# True -> 6 pairs per arm, GRID 8. Flip to False only AFTER the smoke reads.
SMOKE = True

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
import sys, time, json, gc
from pathlib import Path
import numpy as np, pandas as pd, torch

REPO = Path.cwd()
while not (REPO / "src" / "frame").is_dir():
    assert REPO != REPO.parent, "run me from inside the repo"
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "experiments" / "29-sam2-temporal" / "_models"))

EXP = REPO / "experiments" / "29-sam2-temporal"
TAG = "smoke" if SMOKE else "full"
OUT = EXP / "runs" / "step6_sam2_temporal" / TAG
OUT.mkdir(parents=True, exist_ok=True)

if SMOKE:
    GRID = 8

DATA_ROOT = next((d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
                  if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"
assert Path(SAM2, "config.json").exists(), f"SAM 2 checkpoint not at {SAM2}"
print(f"OK    data_root {DATA_ROOT}")
print(f"OK    sam2      {SAM2}")
print(f"OK    out       {OUT}")
print(f"      gap<={GAP_MAX}s gold>={MIN_GOLD} jump>={JUMP} grid={GRID} smoke={SMOKE}")

In [ ]:
# --- the population: consecutive same-video pairs, both golds attached ---------
import sam2_probe as sp
from frame.config import BaselineConfig
from frame.data import FrameProvider, load_frame_items
from frame.parsing import parse_number

cfg = BaselineConfig(data_root=DATA_ROOT, out_dir=OUT, run_name=TAG, seed=SEED)
items = load_frame_items(cfg, splits=("train", "test"))
print(f"      frame items: {len(items):,}")

def read_gold(ref):
    # `number` only: a `fo_class` or `binary` gold is not a count and differencing it is
    # nonsense. RULES EVAL — the format comes from the reference, never from the string.
    if str(getattr(ref, "_format", "")) != "number":
        return None
    v = parse_number(getattr(ref, "answer", ""))
    return None if v != v else int(v)

pairs = sp.build_pairs(items, gap_max=GAP_MAX, min_gold=MIN_GOLD, jump=JUMP,
                       read_gold=read_gold)
assert not pairs.empty, "no pairs — check read_gold and the gap/gold thresholds"
print(pairs.arm.value_counts().to_string())

sel = sp.match_arms(pairs, seed=SEED)
sel = sel[sel.arm.isin(("J", "S"))]
if SMOKE:
    sel = pd.concat([g.head(6) for _, g in sel.groupby("arm")])
sel.to_csv(OUT / "pairs.csv", index=False)
print(f"\n      using {len(sel)} pairs: " +
      ", ".join(f"{a}={n}" for a, n in sel.arm.value_counts().items()))
assert {"J", "S"} <= set(sel.arm), "one arm is empty — no comparison is possible"
print(f"      gap  J={sel[sel.arm=='J'].gap_s.mean():.3f}s  "
      f"S={sel[sel.arm=='S'].gap_s.mean():.3f}s  (matched)")

In [ ]:
# --- SAM 2, from our PINNED transformers — no new dependency --------------------
from transformers import Sam2VideoModel, Sam2VideoProcessor

proc = Sam2VideoProcessor.from_pretrained(SAM2)
model = Sam2VideoModel.from_pretrained(SAM2, dtype=torch.bfloat16).to("cuda").eval()
print("OK    SAM 2 loaded (transformers Sam2VideoModel, sam2.1-hiera-large)")

prov = FrameProvider(cfg)
by_q = {it.request.qID: it for it in items}

def frames_of(row, second="b"):
    """The two frames of a pair, decoded from the source video."""
    a = by_q[row.qID_a]
    prov.ensure_reader(a)
    vr = prov._get_reader(a.dataset, a.video_id)
    ia = int(row.frame_a)
    ib = int(row.frame_b) if second == "b" else min(ia + 1, len(vr) - 1)
    return [np.asarray(vr[ia].asnumpy()), np.asarray(vr[ib].asnumpy())]

def run_pair(row, second="b"):
    fr = frames_of(row, second)
    sess, oids, ma = sp.seed_instances(
        model, proc, fr, frame_idx=0, grid=GRID, min_area_frac=MIN_AREA_FRAC,
        max_area_frac=MAX_AREA_FRAC, nms_iou=NMS_IOU)
    if not oids:
        return {"n_seed": 0, "n_survived": 0, "persistence": float("nan"),
                "mean_iou": float("nan")}
    mb = sp.propagate_pair(model, proc, sess, oids, fr, target_idx=1)
    return sp.pair_stats(ma, mb, iou_survive=IOU_SURVIVE)

In [ ]:
# --- 🔴 BLOCKING C2: can SAM 2 track this footage at all? -----------------------
t0 = time.perf_counter()
adj = [run_pair(r, second="adjacent") for r in sel.head(8).itertuples()]
c2 = sp.control_c2(adj, floor=C2_FLOOR)
print(json.dumps(c2, indent=2))
assert c2["passes"], (
    f"GATE C2 — persistence one frame apart is {c2['persistence_adjacent']:.3f} "
    f"(< {C2_FLOOR}). SAM 2 cannot hold a track across 40 ms of this footage, so neither "
    "arm measures the scene. There is no verdict; the tracker is the finding.")
print(f"\nOK    SAM 2 tracks our footage — the arms are readable ({time.perf_counter()-t0:.0f}s)")

In [ ]:
# --- 🔴 BLOCKING C1: does the seeded count carry ANY count information? ---------
# Rodrigo's negative control, restated as separation (see the README for why the literal
# `K == 1` test cannot be used on a class-agnostic segmenter).
one = pairs[(pairs.gold_a == 1)].head(8 if SMOKE else 40)
high = sel[sel.gold_a >= MIN_GOLD].head(8 if SMOKE else 40)

def seed_count(row):
    fr = frames_of(row)
    _, oids, _ = sp.seed_instances(
        model, proc, fr, frame_idx=0, grid=GRID, min_area_frac=MIN_AREA_FRAC,
        max_area_frac=MAX_AREA_FRAC, nms_iou=NMS_IOU)
    return len(oids)

k_one = [seed_count(r) for r in one.itertuples()]
k_high = [seed_count(r) for r in high.itertuples()]
c1 = sp.control_c1(k_high, k_one, min_sep=C1_MIN_SEPARATION)
print(json.dumps(c1, indent=2))
assert c1["passes"], (
    f"GATE C1 — frames with gold>={MIN_GOLD} seed {c1['mean_k_gold_high']:.2f} instances "
    f"and frames with gold==1 seed {c1['mean_k_gold_one']:.2f} (separation "
    f"{c1['separation']:.2f} < {C1_MIN_SEPARATION}). Nothing SAM emits is count-bearing. "
    "NO VERDICT — the persistence-only fallback is NOT pre-registered and reading it here "
    "would be moving the goalposts. Take it back to Rodrigo, whose amendment this is.")
print("\nOK    the seeded instance count separates high-gold from gold==1 frames")

In [ ]:
# --- the two arms --------------------------------------------------------------
t0 = time.perf_counter()
rows = []
for r in sel.itertuples():
    s = run_pair(r)
    rows.append({"qID_a": r.qID_a, "qID_b": r.qID_b, "arm": r.arm, "gap_s": r.gap_s,
                 "gold_a": r.gold_a, "gold_b": r.gold_b, "delta": r.delta, **s})
per_pair = pd.DataFrame(rows)
per_pair.to_csv(OUT / "per_pair.csv", index=False)
print(f"      {len(per_pair)} pairs in {time.perf_counter()-t0:.0f}s\n")
print(per_pair.groupby("arm")[["n_seed", "persistence", "mean_iou"]].mean().round(4).to_string())

In [ ]:
# --- the pre-registered three-way verdict --------------------------------------
pj = per_pair[per_pair.arm == "J"].persistence.dropna().tolist()
ps = per_pair[per_pair.arm == "S"].persistence.dropna().tolist()
v = sp.verdict(pj, ps, eps=EPS_PERSIST, low=LOW_PERSIST)
v["control_c1"], v["control_c2"] = c1, c2
(OUT / "verdict.json").write_text(json.dumps(v, indent=2))
print(json.dumps(v, indent=2))

mark = {"LABEL": "GREEN", "SCENE": "RED"}.get(v["verdict"], "BLACK")
print(f"\n[{mark}] {v['verdict']} — {v['reading']}")
print("   LABEL      -> the sub-second gold instability is annotation, and `number` has a")
print("                 label ceiling we have been charging to the model.")
print("   SCENE      -> the label is exonerated; the counting deficit is ours to fix.")
print("   NO_VERDICT -> nothing is concluded. Do not write this up as either.")

## Reading it

- **The verdict is the cell above, at the declared thresholds.** The ≤3 s population is
  exploratory and does not decide — only ≤1 s carries the *same physical instance by
  construction* claim that the whole probe rests on.
- **Persistence, not mask count, is the statistic.** SAM 2 is class-agnostic: it segments tissue
  and instruments too, so the number of masks is never the number of foreign objects. C1 exists
  to check that the count is *informative*, not that it is *correct*.
- **If either control fails there is no verdict.** C2 failing means we measured the tracker. C1
  failing means the seeded count is not count-bearing, and the persistence-only fallback is
  deliberately not pre-registered — that decision goes back to the team.
- **A LABEL verdict does not say the gold is garbage.** [[gold-is-signal-model-underuses-it]]
  measures a blind human ordering at r = +0.72 against it, and
  [[model-out-ranks-the-blind-human]] has our own model at 0.8303. The claim available here is
  narrow: what fraction of the **sub-second** instability is annotation.
- **This probe changes no training and ships nothing.** It sets the denominator for how much of
  the `number` deficit is even addressable.